In [1]:
import csv
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.src_data.background_dataset import SARBackgroundDataset
from src.models.cae import ConvolutionalAutoencoder

In [2]:
# Configurations
RANDOM_SEED = 42
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-3
PATIENCE = 7
NUM_WORKERS = 2

In [3]:
# Reproduceability
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
# File/Folder Paths
PATCH_DIR = PROJECT_ROOT / "data/SSDD/background_patches"
SPLIT_DIR = PROJECT_ROOT / "data/SSDD/splits"

TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"

CHECKPOINT_DIR = PROJECT_ROOT / "models/checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs/training"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_cae.pth"

In [5]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [6]:
# Dataset and DataLoaders
train_dataset = SARBackgroundDataset(
    metadata_file=TRAIN_CSV,
    patch_dir=PATCH_DIR,
)

val_dataset = SARBackgroundDataset(
    metadata_file=VAL_CSV,
    patch_dir=PATCH_DIR,
)

if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise RuntimeError("Training or validation dataset is empty. Check the split CSV files and paths.")

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": device.type == "cuda",
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)

print(f"Training patches: {len(train_dataset)}")
print(f"Validation patches: {len(val_dataset)}")

Training patches: 11145
Validation patches: 2775


In [7]:
# Model, Loss, Optimizer
model = ConvolutionalAutoencoder().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

In [ ]:
# Training and Validation Loop
history = {
    "train_loss": [],
    "val_loss": [],
}

best_val_loss = float("inf")
epochs_without_improvement = 0


def run_epoch(loader, training):
    """Run one training or validation epoch."""

    model.train(training)

    total_squared_error = 0.0
    total_pixels = 0

    for images in loader:
        images = images.to(
            device,
            non_blocking=True,
        )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            reconstructions = model(images)
            loss = criterion(reconstructions, images)

            if training:
                loss.backward()
                optimizer.step()

        pixels = images.numel()
        total_squared_error += loss.item() * pixels
        total_pixels += pixels

    return total_squared_error / total_pixels


for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        # Save the best model and enough information to reproduce this training run.
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": best_val_loss,
                "config": {
                    "patch_size": 64,
                    "batch_size": BATCH_SIZE,
                    "learning_rate": LEARNING_RATE,
                    "seed": RANDOM_SEED,
                },
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] "
        f"train_loss={train_loss:.6f} "
        f"val_loss={val_loss:.6f}"
        + ("  [best]" if improved else "")
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

Epoch [01/50] train_loss=0.016024 val_loss=0.007194  [best]
Epoch [02/50] train_loss=0.006641 val_loss=0.006531  [best]
Epoch [03/50] train_loss=0.005988 val_loss=0.005842  [best]
Epoch [04/50] train_loss=0.005523 val_loss=0.005487  [best]
Epoch [05/50] train_loss=0.005214 val_loss=0.005302  [best]
Epoch [06/50] train_loss=0.005087 val_loss=0.005369
Epoch [07/50] train_loss=0.005011 val_loss=0.005069  [best]
Epoch [08/50] train_loss=0.004898 val_loss=0.005461
Epoch [09/50] train_loss=0.004776 val_loss=0.005198
Epoch [10/50] train_loss=0.004688 val_loss=0.005058  [best]
Epoch [11/50] train_loss=0.004573 val_loss=0.004582  [best]
Epoch [12/50] train_loss=0.004387 val_loss=0.004393  [best]
Epoch [13/50] train_loss=0.004215 val_loss=0.004396
Epoch [14/50] train_loss=0.004130 val_loss=0.004263  [best]
Epoch [15/50] train_loss=0.004155 val_loss=0.004310
Epoch [16/50] train_loss=0.003971 val_loss=0.004031  [best]
Epoch [17/50] train_loss=0.003900 val_loss=0.004029  [best]
Epoch [18/50] train_

In [9]:
# Saving Loss History
history_path = OUTPUT_DIR / "loss_history.csv"

with open(history_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss"])

    for i, (train_loss, val_loss) in enumerate(
        zip(history["train_loss"], history["val_loss"]),
        start=1,
    ):
        writer.writerow([i, train_loss, val_loss])

In [10]:
# Plotting the Learning Curves
plt.figure(figsize=(9, 5))
plt.plot(history["train_loss"], label="Train MSE")
plt.plot(history["val_loss"], label="Validation MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("CAE Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

loss_plot = OUTPUT_DIR / "loss_curve.png"
plt.savefig(loss_plot, dpi=150)
plt.close()

print("\nTraining complete.")
print(f"Best validation MSE: {best_val_loss:.6f}")
print(f"Best checkpoint: {BEST_MODEL_PATH}")
print(f"Loss history: {history_path}")
print(f"Loss plot: {loss_plot}")


Training complete.
Best validation MSE: 0.002792
Best checkpoint: /media/aryakumar/ACER/ARYAKUMAR-MISHRA/Projects/Machine-Learning/SAR_Anomaly_Detection/models/checkpoints/best_cae.pth
Loss history: /media/aryakumar/ACER/ARYAKUMAR-MISHRA/Projects/Machine-Learning/SAR_Anomaly_Detection/outputs/training/loss_history.csv
Loss plot: /media/aryakumar/ACER/ARYAKUMAR-MISHRA/Projects/Machine-Learning/SAR_Anomaly_Detection/outputs/training/loss_curve.png
